# 5. 拓展：强制使用工具

## 5.1 tool_choice参数说明
bind_tools 可以传递参数 tool_choice ，用于控制是否强制使用工具。

该字段最终会作为 payload 的 tool_choice 字段传递给模型，OpenAI和Deepseek的官方API服务对于tool_choice 的取值做了相同的规定。


* none ：模型不会调用任何工具。
* auto ： 默认值 ，模型可以自主决定不调用或调用任意数量的工具。
* required ：模型必须调用工具，数量不限。

此外， tool_choice 还支持传递 any ，等价于 required 。

In [1]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

from langchain.tools import tool
from langchain.messages import HumanMessage

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'
model_with_tools = model.bind_tools([get_weather], tool_choice="none")
messages = [
    HumanMessage("今天北京天气如何？别瞎编")
]
response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0e49ad761ce0e744006ab7b1f6e66487d1afabfdfd61018b9a', 'summary': [], 'type': 'reasoning', 'encrypted_content': 'gAAAAABqt7H3WTjdCUTr8XmXGWF-D5YjJKUH5J3wviQJOPz2Iq6-uRsrsXxG3iWmcVyCMbJzRF7EDGCWJ9RSRjMPOAvYwhJ1H-4wbE9yf5vppmixq8v0z-PNxcSiLQ9MHbXf24LRWySZh1vGHqhR1gmY_aujmQbxeSpvu9dP2A5KE5ECIm5cRQEnUdy777ldo4s2rSDhnA-ZG7PaThI7egVdLSDfEynoK9nNKsV1pqm5rIepuS9f4wWiblpysURdeG8-lsoQh2mMX93J0yIhiUimq5akomNQT59rDH5HqqpyhCPA8wkVao37RR9q_s90nX8lTK2kDhRfTIT_feHaS7gFPflG56_SmKZ5VGK-28xviloCAXqpUxjNtjZ5zXp_ZRSrbnWnvLbrExp3xHkLe1FgcXEwR2lKLRw7zps-j7C6ifinPXW48o-9OX-yx-lS0Sdsp0rEaQGkaq3Kbl8_VWi57DE_XkKGpt6ObWD8uDgEnP0TSuRS9sWxLLDfp6_C6-YuBHbPdJH20pll_DDFVW7gEbjvB0iQtT9asBWrn4_qk-PZ-E5WjPcBuQmY2SxV_cDMUKtHU6VXxcBQMwS6XJXZru2PiPZcnZ_wMH-R_hQZHash5P9nanJMgPPCNLXcUoLqU0xNcVRq4wecckUflccxEGbHLA5fBa9SR0lq9NDapFh8qykjL1cNljjXD4haOz2exEYgP9ou2BnQu5z1dZE-iyp_Bd3Q8oneBq2sOtI0Hbpzg44UD6SqcCRKa6kr3Gv2vV5SOfxEszHv8EgsaYpYhav